In [1]:
from src.runner import preprocess
preprocess()

Processing Center for Advanced Life Cycle Engineering cells: 100%|██████████| 7/7 [00:01<00:00,  3.80it/s]


0 processed, 7 skipped



Processing Hawaii Natural Energy Institute cells: 100%|██████████| 15/15 [00:04<00:00,  3.52it/s]


0 processed, 15 skipped



Processing Oxford cells: 100%|██████████| 8/8 [00:01<00:00,  4.94it/s]


...

10 processed, 0 skipped



Processing Cell Report Physical Science cells: 100%|██████████| 10/10 [00:00<00:00, 417.10it/s]


10 processed, 0 skipped



Processing Lithos Healthy Data cells: 100%|██████████| 60/60 [00:00<00:00, 1033.36it/s]

60 processed, 0 skipped



In [11]:
# ===== SET HYPERPARAMETERS =======
num_epochs = 3
window_size = 20
window_skip = 1
index_tuple = (1, 2)

In [17]:
# ===== INITIALIZE DATALOADERS =======
from src.data import get_divided_loaders

healthy_train_loader, healthy_test_loader = get_divided_loaders(test_size=0.2, window_size=window_size, window_skip=window_skip, configfile='healthygas', index_tuple=index_tuple)
_, unhealthy_test_loader = get_divided_loaders(test_size=1.0, window_size=window_size, window_skip=window_skip, configfile='unhealthygas', index_tuple=index_tuple)

Checking attributes: 100%|██████████| 392/392 [00:00<00:00, 5461.06it/s]


In [18]:
# ===== INITIALIZE TRANSFORMER AUTOENCODER =======
from src.training import *
from src.models import TransformerAutoencoder
import pickle

DEVICE = torch.device("mps" if torch.mps.is_available() else "cpu")
print("Using device:", DEVICE)

model_filename = f'results/trained_models/GAS-{num_epochs}epoch_{window_size}window_{window_skip}step.pkl'
print("Model:", model_filename)

retrain = True # NOTE: THIS DETERMINES WHETHER YOU LOAD AN EXISTING MODEL OR TRAIN & SAVE A NEW ONE
if retrain:
    transformer_model = TransformerAutoencoder(
        input_dim=len(index_tuple),     # number of features per timestep
        model_dim=64,                   # hidden embedding dimension
        num_heads=4,                    # parallel attention heads
        num_layers=2                    # stacked encoder/decoder layers
    )

    # took 9m 13s to train 1 epoch with 80% of healthy data files
    transformer_model.to(DEVICE)
    for epoch in range(1,num_epochs+1):
        print(f"Epoch {epoch}/{num_epochs}")
        train_loss = train_epoch(transformer_model, healthy_train_loader, DEVICE)
        print(f" Train loss: {train_loss:.6f}")

    with open(model_filename, 'wb') as file:
        pickle.dump(transformer_model, file)

else:
    transformer_model = TransformerAutoencoder.load(model_filename)

Using device: mps
Model: results/trained_models/GAS-3epoch_20window_1step.pkl
Epoch 1/3


Train: 100%|██████████| 40/40 [00:22<00:00,  1.78it/s, train_loss=304]   


 Train loss: 295.877885
Epoch 2/3


Train: 100%|██████████| 40/40 [00:22<00:00,  1.78it/s, train_loss=225]


 Train loss: 218.662108
Epoch 3/3


Train: 100%|██████████| 40/40 [00:22<00:00,  1.78it/s, train_loss=155]

 Train loss: 150.706291
